In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import files
import io
import warnings
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# =================================================================
# BLOQUE 1: RECONSTRUCCIÓN Y CURACIÓN FLEXIBLE DE DATOS
# =================================================================

print("1. Selecciona el archivo CSV (ej. Dataset_Bioinformatics.csv)")
uploaded = files.upload()

if not uploaded:
    print("No se ha seleccionado ningún archivo.")
else:
    file_name = list(uploaded.keys())[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[file_name]), sep=';', encoding='latin-1')

    # Métricas iniciales
    stats = []
    stats.append(['Fase 0: Carga Inicial', len(df_raw)])

    # Pre-procesamiento básico e indispensable
    df_raw['Edad'] = pd.to_numeric(df_raw['Edad'], errors='coerce')
    df_raw['UID'] = df_raw['Centro Médico'].astype(str) + "_" + df_raw['Número de Historia Clínica (IP Paciente)'].astype(str)
    df_raw['Marca temporal'] = pd.to_datetime(df_raw['Marca temporal'], format='mixed')

    # Estandarizamos el texto de las sesiones para evitar fallos por espacios o tildes
    df_raw['Número de sesión'] = df_raw['Número de sesión'].astype(str).str.strip()

    # --- FASE 1: FILTRADO DE DUPLICADOS TÉCNICOS (CORREGIDO Y FLEXIBLE) ---
    # En lugar de eliminar todo si pasa de 1 hora, solo eliminamos si coinciden exactamente en el mismo minuto/segundo
    # o si se enviaron de forma duplicada el mismo día para la misma sesión exacta.
    df_raw['Fecha_Dia'] = df_raw['Marca temporal'].dt.date
    df_clean = df_raw.drop_duplicates(subset=['UID', 'Número de sesión', 'Fecha_Dia'], keep='last').copy()
    stats.append(['Fase 1: Limpieza Duplicados Mismo Día', len(df_clean)])

    # --- FASE 2: CLASIFICACIÓN ROBUSTA DE COHORTES ---
    def classify_cohort(sessions):
        s = set(sessions)
        # Grupo A: Pasó por la fase intermedia del tratamiento activo
        if 'Primera sesión' in s and 'Fase intermedia' in s and 'Final del tratamiento' in s:
            return 'Group A'
        # Grupo B: Protocolo directo pre/post
        elif 'Primera sesión' in s and 'Final del tratamiento' in s:
            return 'Group B'
        return 'Group C'

    cohort_map = df_clean.groupby('UID')['Número de sesión'].apply(classify_cohort).reset_index()
    cohort_map.columns = ['UID', 'Cohort_Group']
    df_final_puro = pd.merge(df_clean, cohort_map, on='UID', how='left')
    stats.append(['Fase 2: Clasificación de Cohortes', len(df_final_puro)])

    # --- IMPRESIÓN DE TABLAS DE CONTROL ---
    df_stats = pd.DataFrame(stats, columns=['Etapa del Proceso', 'Registros Restantes'])
    df_dist = df_final_puro.groupby('Cohort_Group').agg(
        Pacientes_Unicos=('UID', 'nunique'),
        Registros_Totales=('UID', 'size')
    ).reset_index()

    print("\n" + "="*60)
    print("TABLA 1: FLUJO DE CONTROL DE REGISTROS")
    print("="*60)
    print(df_stats.to_string(index=False))

    print("\n" + "="*60)
    print("TABLA 2: DISTRIBUCIÓN POR COHORTES")
    print("="*60)
    print(df_dist.to_string(index=False))


# =================================================================
# BLOQUE 2: SECUENCIACIÓN CRONOLÓGICA DE RECUERDOS EN DATASET PRINCIPAL
# =================================================================

    print("\n" + "="*60)
    print("TABLA 3: ANÁLISIS LONGITUDINAL DE RECUERDOS")
    print("="*60)

    # 1. Ordenamos el dataframe principal COMPLETO por paciente y fecha exacta
    # Esto garantiza que la línea temporal general sea perfecta antes de etiquetar
    df_final_puro = df_final_puro.sort_values(by=['UID', 'Marca temporal'])

    # 2. Identificamos las filas que son "sesión de recuerdo"
    mask_recuerdos = df_final_puro['Número de sesión'].str.lower().str.contains('recuerdo', na=False)

    # 3. Calculamos la secuencia cronológica (1, 2, 3...) agrupando por UID
    secuencia_recuerdos = df_final_puro[mask_recuerdos].groupby('UID').cumcount() + 1

    # 4. Sobrescribimos la etiqueta en la columna 'Número de sesión'
    # Pasará de ser "Sesión de recuerdo" a "1ª Sesión de recuerdo", "2ª Sesión de recuerdo", etc.
    df_final_puro.loc[mask_recuerdos, 'Número de sesión'] = secuencia_recuerdos.astype(str) + "ª Sesión de recuerdo"

    # --- Extracción de métricas para validación visual ---
    df_recuerdos_etiquetados = df_final_puro[mask_recuerdos]

    for grupo in ['Group A', 'Group B']:
        print(f"\n➔ Cohorte: {grupo}")
        print("-" * 45)

        df_grupo = df_recuerdos_etiquetados[df_recuerdos_etiquetados['Cohort_Group'] == grupo]

        if df_grupo.empty:
            print("No se encontraron sesiones de recuerdo para este grupo.")
            continue

        # Contamos pacientes únicos (n) vs registros (N)
        conteo_sesiones = df_grupo.groupby('Número de sesión').agg(
            Pacientes_Unicos_n=('UID', 'nunique'),
            Registros_N=('UID', 'size')
        ).reset_index()

        # Renombramos para claridad en consola respetando n vs N
        conteo_sesiones.columns = ['Sesión', 'Pacientes Únicos (n)', 'Registros Totales (N)']

        print(conteo_sesiones.to_string(index=False))

    print("="*60)


# =================================================================
# BLOQUE 3: EXPORTACIÓN
# =================================================================

    output_csv = 'Dataset_Bioinformatic_Flexible.csv'
    df_final_puro.to_csv(output_csv, index=False, sep=';', encoding='utf-8-sig')
    files.download(output_csv)
    print("\nPROCESO COMPLETADO: Archivo descargado sin filtros agresivos.")

1. Selecciona el archivo CSV (ej. Dataset_Bioinformatics.csv)


Saving Dataset_Bioinformatics_Completo.csv to Dataset_Bioinformatics_Completo.csv

TABLA 1: FLUJO DE CONTROL DE REGISTROS
                    Etapa del Proceso  Registros Restantes
                Fase 0: Carga Inicial                 3568
Fase 1: Limpieza Duplicados Mismo Día                 3549
    Fase 2: Clasificación de Cohortes                 3549

TABLA 2: DISTRIBUCIÓN POR COHORTES
Cohort_Group  Pacientes_Unicos  Registros_Totales
     Group A               449               1509
     Group B               115                258
     Group C              1125               1782

TABLA 3: ANÁLISIS LONGITUDINAL DE RECUERDOS

➔ Cohorte: Group A
---------------------------------------------
               Sesión  Pacientes Únicos (n)  Registros Totales (N)
1ª Sesión de recuerdo                    93                     93
2ª Sesión de recuerdo                    17                     17
3ª Sesión de recuerdo                     9                      9
4ª Sesión de recuerdo      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROCESO COMPLETADO: Archivo descargado sin filtros agresivos.


In [ ]:
# Analizar qué tipos de sesión componen el polémico Grupo C
df_grupo_c = df_final_puro[df_final_puro['Cohort_Group'] == 'Group C']

print("DISTRIBUCIÓN DE SESIONES EN EL GRUPO C:")
print("-" * 40)
print(df_grupo_c['Número de sesión'].value_counts())

print("\n¿CUÁNTOS REGISTROS TIENE CADA PACIENTE EN EL GRUPO C? (Top 10 frecuencias)")
print("-" * 55)
print(df_grupo_c.groupby('UID').size().value_counts().head(10))

DISTRIBUCIÓN DE SESIONES EN EL GRUPO C:
----------------------------------------
Número de sesión
Primera sesión           786
Fase intermedia          550
1ª Sesión de recuerdo    216
Final del tratamiento    107
2ª Sesión de recuerdo     59
3ª Sesión de recuerdo     30
4ª Sesión de recuerdo     15
5ª Sesión de recuerdo     10
6ª Sesión de recuerdo      4
7ª Sesión de recuerdo      3
8ª Sesión de recuerdo      2
Name: count, dtype: int64

¿CUÁNTOS REGISTROS TIENE CADA PACIENTE EN EL GRUPO C? (Top 10 frecuencias)
-------------------------------------------------------
1    610
2    426
3     60
4     17
5      6
6      3
9      1
8      1
7      1
Name: count, dtype: int64
